# Prompt 27: Structured Streaming Fundamentals
## Databricks Certified Associate Developer for Apache Spark
### Topic 5 — Structured Streaming (10%)

---

## What is Structured Streaming?

Structured Streaming is a scalable, fault-tolerant stream processing engine built on top of the Spark SQL engine. The key design principle: **treat a live data stream as a table that is continuously being appended to**.

```
Input stream                  Unbounded input table
──────────────                ───────────────────────────────────
event at t=1   ─────────────► Row added at t=1
event at t=2   ─────────────► Row added at t=2
event at t=3   ─────────────► Row added at t=3  ...ongoing...

You write a batch-style query on this table.
Spark runs it incrementally as new rows arrive.
Result table updates → written to output sink.
```

**Core property:** The DataFrame/Dataset API for streaming is **identical** to the batch API. You use the same `select()`, `filter()`, `groupBy()`, `join()` calls — Spark handles incrementally processing new data.

---

## Input Sources

| Source | `readStream.format(...)` | When to use |
|--------|--------------------------|-------------|
| **Socket** | `'socket'` | Learning / demos only — not fault-tolerant |
| **Rate** | `'rate'` | Load testing / benchmarking — generates rows at a fixed rate |
| **File** | `'csv'`, `'json'`, `'parquet'`, `'text'` | Read new files dropped into a directory |
| **Kafka** | `'kafka'` | Production event streaming |
| **Delta** | `'delta'` (Databricks) | Stream changes from a Delta table |

### Socket source (demo only)
```python
df_socket = spark.readStream \
    .format('socket') \
    .option('host', 'localhost') \
    .option('port', 9999) \
    .load()
# Schema: one column 'value' of StringType
# Start: nc -lk 9999 in another terminal
```

### Rate source (testing / benchmarking)
```python
df_rate = spark.readStream \
    .format('rate') \
    .option('rowsPerSecond', 100) \
    .option('numPartitions', 4) \
    .load()
# Schema: timestamp (TimestampType), value (LongType)
```

### File source (reads new files in a directory)
```python
df_file = spark.readStream \
    .format('json') \
    .schema(schema)          # REQUIRED — schema must be specified explicitly
    .option('maxFilesPerTrigger', 10) \
    .load('/data/incoming/') # watches this directory for new files
# Note: schema() is MANDATORY for file streaming sources
```

### Kafka source
```python
df_kafka = spark.readStream \
    .format('kafka') \
    .option('kafka.bootstrap.servers', 'broker:9092') \
    .option('subscribe', 'my-topic') \
    .option('startingOffsets', 'latest') \
    .load()
# Schema: key, value (both BinaryType), topic, partition, offset, timestamp, timestampType
# Decode value: df_kafka.selectExpr('CAST(value AS STRING)')
```

---

## Output Sinks

| Sink | `writeStream.format(...)` | Notes |
|------|---------------------------|-------|
| **Console** | `'console'` | Debugging only — not production |
| **Memory** | `'memory'` | Debugging — stores in driver memory, queryable as temp view |
| **File** | `'csv'`, `'json'`, `'parquet'` | Append only; writes one file per micro-batch |
| **Kafka** | `'kafka'` | Write streaming results back to Kafka |
| **Delta** | `'delta'` (Databricks) | Production; supports all output modes |
| **foreachBatch** | `writeStream.foreachBatch(func)` | Custom sink — receives each micro-batch as a DataFrame |
| **foreach** | `writeStream.foreach(writer)` | Custom row-level writer |

### foreachBatch pattern (most flexible custom sink)
```python
def process_batch(batch_df, batch_id):
    # batch_df is a regular batch DataFrame — use any batch API
    batch_df.write.mode('append').parquet('/output/results')
    batch_df.write.mode('append').jdbc(url, table, properties)
    # Can write to multiple sinks in one call

query = df_stream.writeStream \
    .foreachBatch(process_batch) \
    .start()
```

---

## Output Modes

Output mode controls **which rows of the result table are written to the sink** each trigger.

| Mode | What gets written | When valid |
|------|-------------------|------------|
| **`append`** (default) | Only NEW rows added since last trigger | Queries with no aggregation; aggregations WITH watermark |
| **`complete`** | The ENTIRE result table every trigger | Aggregations only (result table must fit in memory) |
| **`update`** | Only rows that CHANGED since last trigger | Aggregations; NOT supported by file sink |

### Output mode rules (exam critical)

| Query type | append | complete | update |
|------------|--------|----------|--------|
| No aggregation | ✅ | ❌ | ✅ |
| Aggregation without watermark | ❌ | ✅ | ✅ |
| Aggregation WITH watermark | ✅ | ✅ | ✅ |
| Distinct (deduplication) | ✅ (with watermark) | ❌ | ❌ |

**Why `append` fails without watermark for aggregations:** Spark cannot know when an aggregation is final (might get late data), so it cannot safely append rows.

---

## Trigger Modes

Trigger controls **how often Spark checks for new data and processes a micro-batch**.

```python
from pyspark.sql.streaming import Trigger  # or use string form
```

| Trigger | Code | Behaviour |
|---------|------|----------|
| **Default (micro-batch)** | `.trigger(processingTime='0 seconds')` | Run as fast as possible — next batch starts as soon as previous finishes |
| **Fixed interval** | `.trigger(processingTime='30 seconds')` | Wait at least 30s between batches |
| **Once** | `.trigger(once=True)` | Process all available data in one batch, then stop (**deprecated in 3.3**) |
| **AvailableNow** | `.trigger(availableNow=True)` | Process all available data (multiple batches), then stop — replacement for Once |
| **Continuous** | `.trigger(continuous='1 second')` | Experimental: millisecond latency via a continuous processing engine |

### Trigger syntax options
```python
# 1. Default — run as fast as possible
query = df.writeStream.trigger(processingTime='0 seconds').start()

# 2. Fixed interval
query = df.writeStream.trigger(processingTime='1 minute').start()

# 3. AvailableNow — recommended over Once for Spark 3.3+
query = df.writeStream.trigger(availableNow=True).start()
query.awaitTermination()  # blocks until all data processed

# 4. Continuous (experimental, low-latency)
query = df.writeStream.trigger(continuous='500 milliseconds').start()
```

---

## Starting, Monitoring, and Stopping Streams

```python
# Start a stream — returns a StreamingQuery object
query = df_stream.writeStream \
    .format('console') \
    .outputMode('append') \
    .option('checkpointLocation', '/tmp/checkpoint/') \
    .trigger(processingTime='10 seconds') \
    .queryName('my_stream') \
    .start()

# StreamingQuery methods
query.id            # unique UUID for this query
query.runId         # unique UUID for this *run* (changes on restart)
query.name          # name set with queryName()
query.isActive      # True if still running
query.status        # dict: {message, isDataAvailable, isTriggerActive}
query.recentProgress  # list of recent micro-batch stats
query.lastProgress    # dict of last micro-batch stats (input rows, duration, etc.)
query.exception()   # returns the exception if query failed

# Block driver until stream finishes (or forever if no stopping condition)
query.awaitTermination()
query.awaitTermination(timeout=60)  # wait max 60 seconds

# Stop the stream gracefully
query.stop()

# Access all active streams
spark.streams.active  # list of all active StreamingQuery objects

# Wait for any stream to terminate
spark.streams.awaitAnyTermination()
```

---

## Checkpointing

Checkpointing is **required for production streaming** and **mandatory for `once`/`availableNow` triggers**. It saves query progress and state so the stream can recover from failures.

```python
query = df_stream.writeStream \
    .option('checkpointLocation', '/path/to/checkpoint/') \
    .format('parquet') \
    .outputMode('append') \
    .start('/path/to/output/')
```

What checkpointing saves:
- **Offsets** — which data has been read (e.g., Kafka offsets, file list)
- **State** — stateful aggregation results (for windowed queries)
- **Commit log** — which micro-batches have been committed to the sink

---

## isStreaming Property

```python
df_batch = spark.read.parquet('/data/')
df_stream = spark.readStream.parquet('/data/')

print(df_batch.isStreaming)   # False
print(df_stream.isStreaming)  # True
```

---

## Classic Example: Streaming Word Count

```python
from pyspark.sql.functions import explode, split

# Read from socket (demo only)
lines = spark.readStream \
    .format('socket') \
    .option('host', 'localhost') \
    .option('port', 9999) \
    .load()

# Split lines into words — identical to batch API
words = lines.select(explode(split(col('value'), ' ')).alias('word'))

# Count words
word_counts = words.groupBy('word').count()

# Write to console with complete mode (entire count table each trigger)
query = word_counts.writeStream \
    .outputMode('complete') \
    .format('console') \
    .trigger(processingTime='5 seconds') \
    .start()

query.awaitTermination()
```

In [ ]:
# Cell 1: Setup
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, explode, split, window, count, sum as _sum,
    current_timestamp, to_timestamp, lit, expr, avg
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, TimestampType, LongType
)
import os, shutil, time, threading

spark = SparkSession.builder \
    .appName('StructuredStreamingFundamentals') \
    .master('local[2]') \
    .config('spark.sql.shuffle.partitions', '2') \
    .getOrCreate()

spark.sparkContext.setLogLevel('ERROR')

# Checkpoint and output dirs (clean up before each run)
CHECKPOINT_DIR = '/tmp/spark_stream_checkpoints'
OUTPUT_DIR     = '/tmp/spark_stream_output'
for d in [CHECKPOINT_DIR, OUTPUT_DIR]:
    if os.path.exists(d):
        shutil.rmtree(d)
    os.makedirs(d)

print('Setup complete')

In [ ]:
# Cell 2: Rate source — explore schema and use for demos

df_rate = spark.readStream \
    .format('rate') \
    .option('rowsPerSecond', 10) \
    .option('numPartitions', 2) \
    .load()

print('isStreaming:', df_rate.isStreaming)   # True
print()
print('Rate source schema:')
df_rate.printSchema()
# root
#  |-- timestamp: timestamp (nullable = true)
#  |-- value: long (nullable = true)

# Write to memory sink for inspection
query_rate = df_rate.writeStream \
    .format('memory') \
    .queryName('rate_demo') \
    .outputMode('append') \
    .trigger(processingTime='2 seconds') \
    .start()

# Let it run for a few seconds to collect some rows
time.sleep(6)

print('\nSample rows from rate source (via memory sink):')
spark.sql('SELECT * FROM rate_demo ORDER BY value').show(10)

print('\nStream status:', query_rate.status)
print('Last progress input rows:', query_rate.lastProgress.get('numInputRows') if query_rate.lastProgress else 'N/A')

query_rate.stop()
print('\nStream stopped. isActive:', query_rate.isActive)

In [ ]:
# Cell 3: File source — stream new files from a directory

STREAM_INPUT_DIR = '/tmp/spark_file_stream_input'
FILE_CHECKPOINT  = '/tmp/spark_file_stream_ckpt'
for d in [STREAM_INPUT_DIR, FILE_CHECKPOINT]:
    if os.path.exists(d):
        shutil.rmtree(d)
    os.makedirs(d)

# Define schema (REQUIRED for file streaming sources)
file_schema = StructType([
    StructField('event_id',   IntegerType(), True),
    StructField('user_id',    IntegerType(), True),
    StructField('event_type', StringType(),  True),
    StructField('amount',     DoubleType(),  True),
])

# Set up file streaming reader
df_files = spark.readStream \
    .format('json') \
    .schema(file_schema) \
    .option('maxFilesPerTrigger', 2) \
    .load(STREAM_INPUT_DIR)

print('File stream isStreaming:', df_files.isStreaming)
df_files.printSchema()

# Write aggregation to memory sink
df_agg = df_files.groupBy('event_type').agg(
    count('event_id').alias('event_count'),
    _sum('amount').alias('total_amount')
)

query_files = df_agg.writeStream \
    .format('memory') \
    .queryName('file_stream_agg') \
    .outputMode('complete') \
    .option('checkpointLocation', FILE_CHECKPOINT) \
    .trigger(processingTime='2 seconds') \
    .start()

time.sleep(3)

# Drop a batch of files
import json
batch1 = [
    {'event_id': 1, 'user_id': 10, 'event_type': 'click',    'amount': 0.0},
    {'event_id': 2, 'user_id': 11, 'event_type': 'purchase', 'amount': 29.99},
    {'event_id': 3, 'user_id': 12, 'event_type': 'click',    'amount': 0.0},
]
with open(os.path.join(STREAM_INPUT_DIR, 'batch1.json'), 'w') as f:
    for row in batch1:
        f.write(json.dumps(row) + '\n')

time.sleep(4)  # let stream process the file
print('\nAfter batch 1:')
spark.sql('SELECT * FROM file_stream_agg').show()

# Drop another file
batch2 = [
    {'event_id': 4, 'user_id': 13, 'event_type': 'purchase', 'amount': 59.99},
    {'event_id': 5, 'user_id': 14, 'event_type': 'view',     'amount': 0.0},
]
with open(os.path.join(STREAM_INPUT_DIR, 'batch2.json'), 'w') as f:
    for row in batch2:
        f.write(json.dumps(row) + '\n')

time.sleep(4)
print('\nAfter batch 2:')
spark.sql('SELECT * FROM file_stream_agg').show()

query_files.stop()
print('\nFile stream stopped')

In [ ]:
# Cell 4: Output modes comparison — append vs complete vs update

print('=== Output mode demonstration ===')
print()

# Use the rate source for all three demos
def make_rate_stream():
    return spark.readStream \
        .format('rate') \
        .option('rowsPerSecond', 5) \
        .load() \
        .withColumn('category', expr("CASE WHEN value % 3 = 0 THEN 'A' WHEN value % 3 = 1 THEN 'B' ELSE 'C' END"))

# -------------------------------------------------------
# 1. APPEND mode — no aggregation
#    Each trigger writes ONLY the new rows added that trigger
# -------------------------------------------------------
print('--- 1. APPEND mode (no aggregation) ---')
ckpt1 = '/tmp/ckpt_append'
if os.path.exists(ckpt1): shutil.rmtree(ckpt1)

q_append = make_rate_stream() \
    .filter(col('value') % 2 == 0) \
    .writeStream \
    .format('memory') \
    .queryName('demo_append') \
    .outputMode('append') \
    .option('checkpointLocation', ckpt1) \
    .trigger(processingTime='2 seconds') \
    .start()

time.sleep(7)
append_count = spark.sql('SELECT COUNT(*) as n FROM demo_append').collect()[0]['n']
print(f'Append mode — rows accumulated: {append_count} (grows over time)')
spark.sql('SELECT * FROM demo_append ORDER BY value DESC').show(5)
q_append.stop()

# -------------------------------------------------------
# 2. COMPLETE mode — aggregation
#    Each trigger writes the ENTIRE result table
# -------------------------------------------------------
print('--- 2. COMPLETE mode (aggregation — entire result each trigger) ---')
ckpt2 = '/tmp/ckpt_complete'
if os.path.exists(ckpt2): shutil.rmtree(ckpt2)

q_complete = make_rate_stream() \
    .groupBy('category') \
    .agg(count('value').alias('cnt'), _sum('value').alias('total')) \
    .writeStream \
    .format('memory') \
    .queryName('demo_complete') \
    .outputMode('complete') \
    .option('checkpointLocation', ckpt2) \
    .trigger(processingTime='2 seconds') \
    .start()

time.sleep(7)
print('Complete mode — full result table (3 rows, one per category):')
spark.sql('SELECT * FROM demo_complete ORDER BY category').show()
q_complete.stop()

# -------------------------------------------------------
# 3. UPDATE mode — aggregation
#    Each trigger writes only CHANGED rows
# -------------------------------------------------------
print('--- 3. UPDATE mode (aggregation — only changed rows) ---')
ckpt3 = '/tmp/ckpt_update'
if os.path.exists(ckpt3): shutil.rmtree(ckpt3)

q_update = make_rate_stream() \
    .groupBy('category') \
    .agg(count('value').alias('cnt')) \
    .writeStream \
    .format('console') \
    .outputMode('update') \
    .option('checkpointLocation', ckpt3) \
    .trigger(processingTime='2 seconds') \
    .start()

time.sleep(6)  # observe 3 micro-batch outputs in console — each shows only changed rows
q_update.stop()

print('\nOutput modes summary:')
print('  append   — new rows only (no aggregation, or aggregation + watermark)')
print('  complete — entire result table (aggregations only)')
print('  update   — changed rows only (aggregations; not supported by file sink)')

In [ ]:
# Cell 5: Trigger modes and foreachBatch sink

print('=== Trigger modes ===')

# -------------------------------------------------------
# Trigger.AvailableNow — process all available data then stop
# -------------------------------------------------------
AVAIL_INPUT = '/tmp/spark_avail_now_input'
AVAIL_CKPT  = '/tmp/spark_avail_now_ckpt'
for d in [AVAIL_INPUT, AVAIL_CKPT]:
    if os.path.exists(d): shutil.rmtree(d)
    os.makedirs(d)

avail_schema = StructType([
    StructField('id',    IntegerType(), True),
    StructField('value', DoubleType(),  True),
])

# Pre-create several files
for i in range(3):
    rows = [{'id': i*10+j, 'value': float(i*10+j) * 1.5} for j in range(5)]
    with open(os.path.join(AVAIL_INPUT, f'file_{i}.json'), 'w') as f:
        for row in rows:
            f.write(json.dumps(row) + '\n')

df_avail = spark.readStream \
    .format('json') \
    .schema(avail_schema) \
    .option('maxFilesPerTrigger', 1) \
    .load(AVAIL_INPUT)

q_avail = df_avail.writeStream \
    .format('memory') \
    .queryName('avail_now') \
    .outputMode('append') \
    .option('checkpointLocation', AVAIL_CKPT) \
    .trigger(availableNow=True) \
    .start()

q_avail.awaitTermination(timeout=30)  # blocks until all data processed

print('AvailableNow trigger — all data processed, stream stopped automatically')
print(f'isActive: {q_avail.isActive}')  # False — stream stopped itself
print(f'Total rows processed: {spark.sql("SELECT COUNT(*) as n FROM avail_now").collect()[0]["n"]}')
spark.sql('SELECT * FROM avail_now ORDER BY id').show(20)

# -------------------------------------------------------
# foreachBatch sink — write to multiple destinations per batch
# -------------------------------------------------------
print('\n=== foreachBatch sink ===')

FB_INPUT = '/tmp/spark_fb_input'
FB_CKPT  = '/tmp/spark_fb_ckpt'
FB_OUTPUT= '/tmp/spark_fb_output'
for d in [FB_INPUT, FB_CKPT, FB_OUTPUT]:
    if os.path.exists(d): shutil.rmtree(d)
    os.makedirs(d)

with open(os.path.join(FB_INPUT, 'data.json'), 'w') as f:
    for i in range(20):
        f.write(json.dumps({'id': i, 'category': 'A' if i % 2 == 0 else 'B', 'amount': float(i * 5)}) + '\n')

fb_schema = StructType([
    StructField('id',       IntegerType(), True),
    StructField('category', StringType(),  True),
    StructField('amount',   DoubleType(),  True),
])

batch_results = []

def process_batch(batch_df, batch_id):
    print(f'[foreachBatch] batch_id={batch_id}, rows={batch_df.count()}')
    # batch_df is a regular batch DataFrame — full batch API available
    batch_df.write.mode('append').parquet(FB_OUTPUT)
    # Also compute and collect aggregation for logging
    agg = batch_df.groupBy('category').agg(count('id').alias('n'), _sum('amount').alias('total'))
    batch_results.append({'batch_id': batch_id, 'rows': batch_df.count()})
    agg.show()

df_fb = spark.readStream \
    .format('json') \
    .schema(fb_schema) \
    .load(FB_INPUT)

q_fb = df_fb.writeStream \
    .foreachBatch(process_batch) \
    .option('checkpointLocation', FB_CKPT) \
    .trigger(availableNow=True) \
    .start()

q_fb.awaitTermination(timeout=30)

print('\nforeachBatch complete. Verifying parquet output...')
df_output = spark.read.parquet(FB_OUTPUT)
print(f'Rows written to parquet: {df_output.count()}')
df_output.show(5)

spark.stop()
print('\nDone.')

## Real-World Scenarios

<details>
<summary>Scenario 1: Choosing the correct output mode for a streaming word count</summary>

**Situation:**
A developer implements a streaming word count using `groupBy('word').count()` and tries to use `outputMode('append')`. The query fails with:
`AnalysisException: Append output mode not supported when there are streaming aggregations on streaming DataFrames/Datasets without watermark.`

**Explanation:**
With `groupBy().count()`, the count for a word can increase on any future micro-batch. Spark cannot mark a result row as "final" without a watermark, so it cannot safely *append* new rows (it would need to update existing ones). The result table must be written in its entirety each trigger.

**Fix:**
```python
# Option A: complete mode — write entire count table every trigger
word_counts.writeStream \
    .outputMode('complete') \
    .format('console') \
    .start()

# Option B: update mode — write only words whose count changed
word_counts.writeStream \
    .outputMode('update') \
    .format('console') \
    .start()

# Option C: use watermark to enable append mode
from pyspark.sql.functions import current_timestamp
lines.withColumn('event_time', current_timestamp()) \
     .withWatermark('event_time', '10 minutes') \
     .groupBy(window('event_time', '5 minutes'), col('word')) \
     .count() \
     .writeStream.outputMode('append').start()
```

**Exam Sub-topic:** output mode rules; when `append` requires a watermark for aggregations
</details>

<details>
<summary>Scenario 2: Processing files that arrive in a directory using the file streaming source</summary>

**Situation:**
An IoT pipeline receives JSON files dropped to an S3 bucket every minute. A developer needs to read them in near-real-time, aggregate by device type, and write results to Parquet.

**Solution:**
```python
device_schema = StructType([
    StructField('device_id',   StringType(),    True),
    StructField('device_type', StringType(),    True),
    StructField('reading',     DoubleType(),    True),
    StructField('ts',          TimestampType(), True),
])

df_devices = spark.readStream \
    .format('json') \
    .schema(device_schema) \
    .option('maxFilesPerTrigger', 5) \
    .load('s3://my-bucket/iot-incoming/')

df_agg = df_devices \
    .groupBy('device_type') \
    .agg(avg('reading').alias('avg_reading'), count('device_id').alias('count'))

query = df_agg.writeStream \
    .format('parquet') \
    .outputMode('complete') \
    .option('checkpointLocation', 's3://my-bucket/checkpoints/iot-agg') \
    .option('path', 's3://my-bucket/iot-aggregated/') \
    .trigger(processingTime='1 minute') \
    .start()
```

**Key points:** Schema must be specified; `maxFilesPerTrigger` controls backfill rate; checkpoint is required for file source recovery.

**Exam Sub-topic:** file source schema requirement; checkpoint; `maxFilesPerTrigger`; output mode for aggregation
</details>

<details>
<summary>Scenario 3: Reading from Kafka, parsing JSON payload, writing aggregations back to Kafka</summary>

**Situation:**
A developer reads click events from a Kafka topic, parses the JSON value, counts clicks per page, and writes the top pages to another Kafka topic.

**Solution:**
```python
from pyspark.sql.functions import from_json, to_json, struct

event_schema = StructType([
    StructField('user_id',   StringType(), True),
    StructField('page',      StringType(), True),
    StructField('click_ts',  StringType(), True),
])

# Read from Kafka
df_raw = spark.readStream \
    .format('kafka') \
    .option('kafka.bootstrap.servers', 'broker:9092') \
    .option('subscribe', 'click-events') \
    .option('startingOffsets', 'latest') \
    .load()

# Decode and parse
df_parsed = df_raw \
    .selectExpr('CAST(value AS STRING) as json_str') \
    .select(from_json(col('json_str'), event_schema).alias('data')) \
    .select('data.*')

# Aggregate
df_page_counts = df_parsed.groupBy('page').count()

# Write back to Kafka — key and value must both be strings/bytes
df_page_counts \
    .select(
        col('page').cast('string').alias('key'),
        to_json(struct('page', 'count')).alias('value')
    ) \
    .writeStream \
    .format('kafka') \
    .outputMode('update') \
    .option('kafka.bootstrap.servers', 'broker:9092') \
    .option('topic', 'page-counts') \
    .option('checkpointLocation', '/checkpoints/page-counts') \
    .start()
```

**Exam Sub-topic:** Kafka source schema (key/value as binary); `CAST(value AS STRING)`; `from_json`; Kafka sink key/value requirements; `update` output mode
</details>

<details>
<summary>Scenario 4: Using foreachBatch to write to a JDBC sink (which has no native streaming support)</summary>

**Situation:**
A team needs to write streaming aggregation results to a PostgreSQL database, but PostgreSQL has no native Spark streaming sink.

**Solution:**
```python
jdbc_url = 'jdbc:postgresql://db:5432/analytics'
jdbc_props = {'user': 'spark', 'password': 'secret', 'driver': 'org.postgresql.Driver'}

def write_to_postgres(batch_df, batch_id):
    # Use idempotent write — upsert by batch_id or use overwrite for aggregation tables
    batch_df.write \
        .mode('overwrite') \
        .option('dbtable', 'page_click_counts') \
        .jdbc(jdbc_url, 'page_click_counts', properties=jdbc_props)
    print(f'Batch {batch_id}: wrote {batch_df.count()} rows to PostgreSQL')

df_page_counts.writeStream \
    .foreachBatch(write_to_postgres) \
    .outputMode('complete') \
    .option('checkpointLocation', '/checkpoints/pg-page-counts') \
    .trigger(processingTime='30 seconds') \
    .start()
```

**Key insight:** `foreachBatch` receives a standard batch DataFrame — you can use **any batch API or sink** inside it. This is the recommended pattern for writing to sinks that don't have native streaming support.

**Exam Sub-topic:** `foreachBatch` function signature `(batch_df, batch_id)`; use any batch API inside; idempotent writes; supports `complete` output mode
</details>

<details>
<summary>Scenario 5: Using AvailableNow trigger for incremental batch processing (replacing Once)</summary>

**Situation:**
A scheduled job runs every hour to process all new files since the last run, then exits. The team was using `trigger(once=True)` but is migrating to Spark 3.3+.

**Before (Spark 3.2 and earlier):**
```python
query = df_stream.writeStream \
    .trigger(once=True) \
    .option('checkpointLocation', '/ckpt/hourly') \
    .format('delta') \
    .outputMode('append') \
    .start('/data/output')
query.awaitTermination()
```

**After (Spark 3.3+, recommended):**
```python
query = df_stream.writeStream \
    .trigger(availableNow=True) \
    .option('checkpointLocation', '/ckpt/hourly') \
    .format('delta') \
    .outputMode('append') \
    .start('/data/output')
query.awaitTermination()  # waits until all available data is processed
```

**Difference from `once`:** `once=True` processes all data in a single micro-batch (may be huge). `availableNow=True` processes data across multiple optimally-sized micro-batches, then stops — more efficient and less likely to OOM.

**Checkpoint behaviour:** Both `once` and `availableNow` use the checkpoint to track which data was already processed, so only NEW data is processed on each scheduled run.

**Exam Sub-topic:** `trigger(once=True)` deprecated in 3.3; `trigger(availableNow=True)` replacement; `awaitTermination()` for scheduled jobs; checkpoint enables incremental processing
</details>

## Exam Cheat Sheet

| Question | Answer |
|----------|--------|
| Streaming vs batch API | Identical — same `select()`, `filter()`, `groupBy()`, `join()` calls |
| Check if DataFrame is streaming | `df.isStreaming` → True/False |
| Rate source schema | `timestamp` (TimestampType) + `value` (LongType) |
| Kafka source schema | `key`, `value` (both BinaryType), `topic`, `partition`, `offset`, `timestamp` |
| Decode Kafka value | `.selectExpr('CAST(value AS STRING)')` |
| Schema required for file sources? | **Yes — always** (`readStream.schema(my_schema)`) |
| Socket source fault-tolerant? | **No** — demo/testing only |
| `append` mode — what gets written | Only NEW rows since last trigger |
| `complete` mode — what gets written | Entire result table every trigger |
| `update` mode — what gets written | Only changed rows since last trigger |
| `append` valid for aggregation without watermark? | **No** — AnalysisException |
| `complete` valid for no-aggregation query? | **No** |
| `update` supported by file sink? | **No** |
| Default trigger | `processingTime='0 seconds'` — run as fast as possible |
| `trigger(once=True)` — status in 3.3+ | **Deprecated** — use `availableNow=True` |
| Difference: `once` vs `availableNow` | `once` = one giant batch; `availableNow` = multiple optimal batches, then stop |
| Checkpoint required for what? | Production, stateful queries, `once`/`availableNow`, Kafka offset tracking |
| Start a stream | `df.writeStream...start()` → returns `StreamingQuery` |
| Block until stream stops | `query.awaitTermination()` |
| Stop a stream | `query.stop()` |
| `foreachBatch` signature | `def f(batch_df: DataFrame, batch_id: int)` |
| `foreachBatch` use case | Custom/multi-sink; write to JDBC, multiple outputs, apply batch-only APIs |

---

## Practice Q&A

<details>
<summary>Q1: What output mode should you use for a streaming query with no aggregation (just filter and select)?</summary>

**A:** `append` mode. Since there is no aggregation, each new event row is independent — once written it can never change. Spark can safely write only the new rows each trigger.

`complete` mode is invalid for queries without aggregations — it requires an aggregation to define what the "complete result table" is.
`update` mode also works for no-aggregation queries (same behaviour as append when no state changes), but `append` is the conventional choice.
</details>

<details>
<summary>Q2: A streaming query reads from a file source. What happens if you omit .schema()?</summary>

**A:** The query fails with an `AnalysisException`:
`Schema must be specified when creating a streaming source DataFrame. If some files don't have a schema...`

Unlike batch reads where Spark can infer the schema by reading the files upfront, streaming sources cannot read ahead to infer the schema — new files may arrive later. The schema **must** be declared explicitly when using file streaming sources (`json`, `csv`, `parquet`, `avro`).

```python
# Correct:
schema = StructType([StructField('id', IntegerType()), ...])
spark.readStream.format('json').schema(schema).load('/path/to/dir/')
```
</details>

<details>
<summary>Q3: What is the difference between trigger(once=True) and trigger(availableNow=True)?</summary>

**A:**
| | `once=True` | `availableNow=True` |
|-|-------------|---------------------|
| Introduced | Spark 2.0 | Spark 3.3 |
| Status | Deprecated in Spark 3.3 | Recommended replacement |
| Behaviour | Processes all available data in **one single micro-batch**, then stops | Processes all available data across **multiple optimally-sized micro-batches**, then stops |
| Memory risk | High — one massive batch can OOM | Lower — batches are size-controlled |
| Speed | Potentially faster for small data | More efficient for large backlogs |
| Both use checkpoint? | Yes — tracks which data was processed | Yes — same checkpoint format |

Both are used for scheduled ("micro-batch on demand") patterns — run the stream, process new data since last run, exit.
</details>

<details>
<summary>Q4: What does the foreachBatch function signature look like, and what can you do inside it that you cannot do with a regular sink?</summary>

**A:** Signature:
```python
def process_batch(batch_df: DataFrame, batch_id: int) -> None:
    ...
```
- `batch_df` — a regular **batch** DataFrame containing the micro-batch data
- `batch_id` — a monotonically increasing integer identifying the micro-batch

What you can do inside `foreachBatch` that regular sinks don't support:
1. **Write to multiple sinks** in one call (e.g., Parquet + JDBC + another Kafka topic)
2. **Use any batch API** — batch-only transformations, `write.jdbc()`, `write.format('delta')`
3. **Apply idempotency logic** — check `batch_id` to avoid re-processing on retry
4. **Call actions** like `count()`, `collect()` to gather metrics
5. **Write to sinks without native streaming support** (JDBC, Cassandra, custom systems)
</details>

<details>
<summary>Q5: What is the schema of data read from a Kafka streaming source, and how do you extract the message payload?</summary>

**A:** A Kafka streaming source always produces a fixed schema regardless of the message content:

| Column | Type | Description |
|--------|------|-------------|
| `key` | BinaryType | Message key |
| `value` | BinaryType | Message payload |
| `topic` | StringType | Kafka topic name |
| `partition` | IntegerType | Kafka partition number |
| `offset` | LongType | Offset within partition |
| `timestamp` | TimestampType | Message timestamp |
| `timestampType` | IntegerType | 0=CreateTime, 1=LogAppendTime |

**To extract the payload:**
```python
# Step 1: Cast binary to string
df_str = df_kafka.selectExpr('CAST(key AS STRING) as key', 'CAST(value AS STRING) as value')

# Step 2: If value is JSON, parse with from_json
from pyspark.sql.functions import from_json
payload_schema = StructType([StructField('user_id', StringType()), ...])
df_parsed = df_str.select(from_json(col('value'), payload_schema).alias('data')).select('data.*')
```
</details>